# Small Object Detection from Satellite Imagery — Paper Replication
### Yuan et al. (2026), *Expert Systems With Applications* 307, 131061

---

This notebook replicates the methodology of the paper using the **xView**
dataset and **YOLOv11** (the paper's best-performing detector).

| Paper metric (xView) | YOLOv11 | RT-DETR | Faster R-CNN | SSD |
|---|---|---|---|---|
| AP50 | 69.47 | 68.20 | 61.65 | 13.15 |
| AP50:95 | 29.58 | 28.15 | 23.32 | 3.35 |
| F1 (50) | 73.97 | 72.97 | 69.44 | 25.89 |

**Runs on**: Kaggle notebooks, Google Colab, or a local machine with GPU.
Set **one variable** (`RUN_ENV`) in the next section to get started.


---
## 1. User Configuration

Edit the variables in the cell below and then run the entire notebook.

| Variable | Purpose |
|---|---|
| `RUN_ENV` | `"kaggle"`, `"colab"`, or `"local"` |
| `DATASET_HANDLE` | Kaggle dataset slug (do not change unless the dataset moves) |
| `LOCAL_DATASET_DIR` | Used only when `RUN_ENV = "local"` — path to an existing xView folder |
| `LOCAL_KAGGLE_JSON` | Path to `kaggle.json` when running outside Kaggle |
| `MOUNT_GDRIVE` | Colab only — mount Google Drive for persistent saves |
| `GDRIVE_OUTPUT_DIR` | Where on Drive to mirror outputs |
| `FORCE_REDOWNLOAD` | Re-download even if the dataset already exists |
| `RESUME_TRAINING` | Pick up from the latest checkpoint instead of epoch 0 |
| `YOLO_VARIANT` | `"yolo11n.pt"` (fast) / `"yolo11s.pt"` / `"yolo11m.pt"` (paper-scale) |


In [ ]:
# ==========================================================================
#  1.1   EDIT THESE VARIABLES
# ==========================================================================

# ---- Environment ---------------------------------------------------------
RUN_ENV = "kaggle"            # "kaggle" | "colab" | "local"

# ---- Dataset source (same public Kaggle dataset in all environments) -----
DATASET_HANDLE = "hassanmojab/xview-dataset"

# ---- Local-only settings -------------------------------------------------
LOCAL_DATASET_DIR  = "./data/xview"          # ignored unless RUN_ENV="local"
LOCAL_KAGGLE_JSON  = "~/.kaggle/kaggle.json" # path to kaggle.json

# ---- Colab-only settings -------------------------------------------------
MOUNT_GDRIVE       = True                    # mount Google Drive for saves?
GDRIVE_OUTPUT_DIR  = "MyDrive/xview_outputs" # relative to /content/drive/

# ---- Behaviour flags -----------------------------------------------------
FORCE_REDOWNLOAD   = False      # True = delete & re-download the dataset
RESUME_TRAINING    = True       # True = resume from last checkpoint if found
SKIP_TRAINING      = False      # True = skip training, evaluate only

# ---- Model variant (Paper Table 14: YOLOv11 has 25.37 M params) ----------
YOLO_VARIANT = "yolo11n.pt"    # "yolo11n.pt" | "yolo11s.pt" | "yolo11m.pt"

# ---- Paper-aligned hyperparameters (Table 5 — do not change casually) ----
TILE_SIZE      = 512
TRAIN_RATIO    = 0.70
BATCH_SIZE     = 8             # reduce to 4 or 2 if GPU runs OOM
LEARNING_RATE  = 1.01e-4
MOMENTUM       = 0.89
WEIGHT_DECAY   = 5e-4
EPOCHS         = 100
IMG_SIZE       = 512
FLIP_PROB      = 0.5
CONF_THRESHOLD = 0.25
SEED           = 42

# ---- Small-object definition (Paper Section 3.1) -------------------------
MAX_OBJECT_PIXELS = 1000       # upper bound (bounding-box area)
MIN_OBJECT_PIXELS = 10         # lower bound — below this not recognisable

print("Configuration loaded.  RUN_ENV =", RUN_ENV)


---
## 2. Environment Bootstrap

This section:
1. validates `RUN_ENV`
2. installs missing packages
3. imports libraries
4. detects GPU / TPU / CPU
5. sets up directory layout


In [ ]:
# ==========================================================================
#  2.1   Validate RUN_ENV
# ==========================================================================
import os, sys, platform, subprocess, shutil
from pathlib import Path

assert RUN_ENV in ("kaggle", "colab", "local"), (
    f"RUN_ENV must be 'kaggle', 'colab', or 'local', got '{RUN_ENV}'"
)

# Quick sanity: warn if the value looks wrong for this machine
if RUN_ENV == "kaggle" and not os.path.exists("/kaggle"):
    print("WARNING: RUN_ENV='kaggle' but /kaggle does not exist.")
    print("         Are you sure you are on Kaggle?")
if RUN_ENV == "colab":
    try:
        import google.colab  # noqa: F401
    except ImportError:
        print("WARNING: RUN_ENV='colab' but google.colab is not importable.")
        print("         Are you sure you are on Colab?")

print(f"RUN_ENV   : {RUN_ENV}")
print(f"Python    : {sys.version.split()[0]}")
print(f"Platform  : {platform.system()} {platform.release()}")


In [ ]:
# ==========================================================================
#  2.2   Install dependencies
# ==========================================================================
def _install(pkg, pip_name=None):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or pkg],
            stdout=subprocess.DEVNULL,
        )

_install("numpy")
_install("pandas")
_install("matplotlib")
_install("seaborn")
_install("cv2",        "opencv-python-headless")
_install("tqdm")
_install("sklearn",    "scikit-learn")
_install("yaml",       "pyyaml")
_install("torch")
_install("torchvision")
_install("ultralytics")
_install("kaggle")

print("All packages ready.")


In [ ]:
# ==========================================================================
#  2.3   Imports
# ==========================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import json
import random
import warnings
import time
import yaml
import zipfile
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
from datetime import datetime

import torch
import torchvision

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi":     100,
    "font.size":      11,
    "axes.titlesize": 13,
})
sns.set_style("whitegrid")

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

from ultralytics import YOLO


In [ ]:
# ==========================================================================
#  2.4   Hardware detection
# ==========================================================================
def detect_hardware():
    """Detect the best available accelerator and print a summary."""
    info = {"device": "cpu", "name": "CPU", "count": 0, "mem_gb": 0}

    # --- TPU (Colab only) ---
    try:
        if "COLAB_TPU_ADDR" in os.environ:
            info["device"] = "tpu"
            info["name"]   = f"TPU @ {os.environ['COLAB_TPU_ADDR']}"
            info["count"]  = 1
            print("TPU detected, but Ultralytics YOLO does not natively")
            print("support TPU.  Falling back to GPU or CPU.")
    except Exception:
        pass

    # --- GPU ---
    if torch.cuda.is_available():
        info["device"] = "cuda"
        info["count"]  = torch.cuda.device_count()
        info["name"]   = torch.cuda.get_device_name(0)
        info["mem_gb"] = torch.cuda.get_device_properties(0).total_mem / 1e9
    # --- MPS (Apple Silicon) ---
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        info["device"] = "mps"
        info["name"]   = "Apple MPS"
        info["count"]  = 1

    print(f"Device   : {info['device'].upper()}")
    print(f"Name     : {info['name']}")
    if info["count"]:
        print(f"Count    : {info['count']}")
    if info["mem_gb"]:
        print(f"Memory   : {info['mem_gb']:.1f} GB")
    print(f"PyTorch  : {torch.__version__}")
    print(f"Ultralytics loaded")

    if info["device"] == "cpu":
        print()
        print("WARNING: No GPU detected.  Training will be extremely slow.")
        print("         Consider using Kaggle or Colab with a GPU runtime.")

    return info

HW = detect_hardware()


In [ ]:
# ==========================================================================
#  2.5   Directory layout
# ==========================================================================
def build_paths(run_env, mount_gdrive, gdrive_output_dir):
    """
    Create all output directories.  On Colab with Drive mounted the
    outputs are mirrored to Drive so they survive session resets.
    """
    # --- Base working directory ---
    if run_env == "kaggle":
        base = Path("/kaggle/working")
        data = Path("/kaggle/working/xview_data")
    elif run_env == "colab":
        base = Path("/content")
        data = Path("/content/xview_data")
    else:
        base = Path(".")
        data = Path(LOCAL_DATASET_DIR).expanduser().resolve()

    out = base / "outputs"

    # On Colab, optionally mirror to Drive
    drive_out = None
    if run_env == "colab" and mount_gdrive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            drive_out = Path("/content/drive") / gdrive_output_dir
            drive_out.mkdir(parents=True, exist_ok=True)
            out = drive_out          # save directly to Drive
            print(f"Google Drive mounted.  Outputs -> {out}")
        except Exception as e:
            print(f"Drive mount failed ({e}).  Using local /content/outputs.")

    paths = {
        "DATA_DIR":        data,
        "OUTPUT_ROOT":     out,
        "CHECKPOINTS_DIR": out / "checkpoints",
        "PLOTS_DIR":       out / "plots",
        "METRICS_DIR":     out / "metrics",
        "PREDICTIONS_DIR": out / "predictions",
        "LOGS_DIR":        out / "logs",
        "TILES_DIR":       out / "tiles",
        "YOLO_DATASET":    out / "yolo_dataset",
    }

    for d in paths.values():
        d.mkdir(parents=True, exist_ok=True)

    print(f"Data dir   : {paths['DATA_DIR']}")
    print(f"Output dir : {paths['OUTPUT_ROOT']}")
    return paths

PATHS = build_paths(RUN_ENV, MOUNT_GDRIVE, GDRIVE_OUTPUT_DIR)


---
## 3. Dataset Loading

The **same** public Kaggle dataset (`hassanmojab/xview-dataset`) is used in
all three environments.  The loader:

1. checks whether the dataset already exists and is valid,
2. downloads only if necessary,
3. extracts any zips,
4. validates the result.

| Environment | How it loads |
|---|---|
| Kaggle | Uses the dataset attached to the notebook (fast, no download) |
| Colab | Downloads via the Kaggle API (needs `kaggle.json` upload) |
| Local | Downloads via the Kaggle API (needs `~/.kaggle/kaggle.json`) |


In [ ]:
# ==========================================================================
#  3.1   Dataset validation helper
# ==========================================================================
def validate_dataset(data_dir):
    """
    Check whether `data_dir` contains a valid xView dataset.
    Returns (is_valid, geojson_path, images_dir).
    """
    data_dir = Path(data_dir)
    if not data_dir.exists():
        return False, None, None

    # --- Search this dir and one level of subdirs ---
    search_roots = [data_dir]
    search_roots += [d for d in data_dir.iterdir() if d.is_dir()]

    for root in search_roots:
        geos = (list(root.glob("*train*.geojson"))
                + list(root.glob("*xView*.geojson"))
                + list(root.glob("*.geojson")))

        img_dir = None
        for name in ["train_images", "images", "train"]:
            d = root / name
            if d.is_dir() and (list(d.glob("*.tif")) or list(d.glob("*.png"))):
                img_dir = d
                break
        # Flat layout: images directly in root
        if img_dir is None and list(root.glob("*.tif")):
            img_dir = root

        if geos and img_dir:
            return True, geos[0], img_dir

    return False, None, None


# Quick pre-check
is_valid, _, _ = validate_dataset(PATHS["DATA_DIR"])
print(f"Dataset pre-check ({'VALID' if is_valid else 'NOT FOUND'}) "
      f"at {PATHS['DATA_DIR']}")


In [ ]:
# ==========================================================================
#  3.2   Kaggle API helper
# ==========================================================================
def ensure_kaggle_credentials(run_env, local_json_path):
    """
    Make sure the Kaggle API can authenticate.

    - Kaggle notebooks: credentials are automatic.
    - Colab: prompt the user to upload kaggle.json if not found.
    - Local: look at LOCAL_KAGGLE_JSON or ~/.kaggle/kaggle.json.
    """
    if run_env == "kaggle":
        return True   # built-in

    kaggle_dir = Path.home() / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"

    if kaggle_json.exists():
        os.chmod(str(kaggle_json), 0o600)
        print(f"Kaggle credentials found : {kaggle_json}")
        return True

    # Try the user-specified path
    src = Path(local_json_path).expanduser()
    if src.exists():
        kaggle_dir.mkdir(exist_ok=True)
        shutil.copy2(src, kaggle_json)
        os.chmod(str(kaggle_json), 0o600)
        print(f"Copied kaggle.json from  : {src}")
        return True

    # Colab: offer an upload widget
    if run_env == "colab":
        print("kaggle.json not found.  Please upload it now.")
        print("  (Get it from https://www.kaggle.com/settings -> API -> "
              "Create New Token)")
        try:
            from google.colab import files
            uploaded = files.upload()           # interactive widget
            for fname, content in uploaded.items():
                kaggle_dir.mkdir(exist_ok=True)
                (kaggle_json).write_bytes(content)
                os.chmod(str(kaggle_json), 0o600)
                print(f"Saved kaggle.json to {kaggle_json}")
                return True
        except Exception as e:
            print(f"Upload failed: {e}")
            return False

    print("ERROR: kaggle.json not found.")
    print(f"  Expected at: {kaggle_json}")
    print(f"  Or set LOCAL_KAGGLE_JSON to its location.")
    return False


In [ ]:
# ==========================================================================
#  3.3   Download & prepare dataset
# ==========================================================================
def download_and_prepare(run_env, dataset_handle, data_dir,
                         force=False, local_json=LOCAL_KAGGLE_JSON):
    """
    Unified dataset loader for all three environments.
    Returns the validated data_dir or raises RuntimeError.
    """
    data_dir = Path(data_dir)

    # --- 1.  Already valid? ---
    if not force:
        ok, geo, imgs = validate_dataset(data_dir)
        if ok:
            print(f"Dataset already valid at {data_dir}")
            print(f"  GeoJSON : {geo}")
            print(f"  Images  : {imgs}  ({len(list(imgs.glob('*.tif')))} tif files)")
            return data_dir

    # --- 2.  On Kaggle, the dataset may be pre-attached ---
    if run_env == "kaggle":
        slug = dataset_handle.split("/")[-1]
        candidates = [
            Path(f"/kaggle/input/{slug}"),
            Path("/kaggle/input"),
        ]
        for c in candidates:
            ok, geo, imgs = validate_dataset(c)
            if ok:
                print(f"Using Kaggle input dataset at {c}")
                return c
        # Not pre-attached → download into working dir
        print("Dataset not pre-attached.  Will download via Kaggle API.")

    # --- 3.  Ensure credentials ---
    if not ensure_kaggle_credentials(run_env, local_json):
        raise RuntimeError(
            "Cannot authenticate with Kaggle.  Provide kaggle.json."
        )

    # --- 4.  Download ---
    if force and data_dir.exists():
        print(f"FORCE_REDOWNLOAD: removing {data_dir}")
        shutil.rmtree(data_dir)

    data_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {dataset_handle} -> {data_dir}")
    print("  This may take a while for a large dataset ...")

    try:
        import kaggle
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files(
            dataset_handle,
            path=str(data_dir),
            unzip=True,
            quiet=False,
        )
    except Exception as e:
        print(f"  Python API failed ({e}), trying CLI ...")
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", dataset_handle,
             "-p", str(data_dir), "--unzip"],
            check=True,
        )

    # --- 5.  Extract any leftover zips ---
    for zp in list(data_dir.rglob("*.zip")):
        print(f"  Extracting {zp.name} ...")
        try:
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(zp.parent)
            zp.unlink()
        except zipfile.BadZipFile:
            print(f"  Skipping bad zip: {zp.name}")

    # --- 6.  Validate ---
    ok, geo, imgs = validate_dataset(data_dir)
    if not ok:
        raise RuntimeError(
            f"Download finished but dataset is invalid at {data_dir}.\n"
            "Check the contents manually."
        )

    print(f"Dataset ready at {data_dir}")
    print(f"  GeoJSON : {geo}")
    print(f"  Images  : {imgs}")
    return data_dir


DATA_DIR = download_and_prepare(
    RUN_ENV, DATASET_HANDLE, PATHS["DATA_DIR"],
    force=FORCE_REDOWNLOAD, local_json=LOCAL_KAGGLE_JSON,
)


In [ ]:
# ==========================================================================
#  3.4   Lock in dataset structure
# ==========================================================================
VALID, GEOJSON_PATH, IMAGES_DIR = validate_dataset(DATA_DIR)

assert VALID, f"Dataset is not valid at {DATA_DIR}"
print(f"GeoJSON   : {GEOJSON_PATH}")
print(f"Images    : {IMAGES_DIR}")
n_tifs = len(list(IMAGES_DIR.glob("*.tif")))
n_pngs = len(list(IMAGES_DIR.glob("*.png")))
print(f"  .tif files : {n_tifs}")
print(f"  .png files : {n_pngs}")


---
## 4. Exploratory Data Analysis

### Paper context (Section 3.1, Table 3)

Four vehicle classes with bounding-box area <= 1 000 pixels:

| Class | ID | Objects (paper) | Size range |
|---|---|---|---|
| Small Car | 18 | 423 092 | 20 – 957 px |
| Passenger Vehicle | 17 | 5 900 | 20 – 735 px |
| Pickup Truck | 20 | 2 008 | 102 – 700 px |
| Utility Truck | 21 | 7 264 | 14 – 840 px |


In [ ]:
# ==========================================================================
#  4.1   Parse GeoJSON
# ==========================================================================
def load_xview_geojson(geojson_path):
    """Parse xView GeoJSON -> DataFrame with pixel bounding boxes."""
    print(f"Loading {geojson_path} ...")
    with open(geojson_path) as f:
        data = json.load(f)

    records = []
    for feat in tqdm(data["features"], desc="Parsing"):
        props  = feat.get("properties", {})
        coords = props.get("bounds_imcoords", "")
        if not coords:
            continue
        try:
            x1, y1, x2, y2 = (float(v) for v in coords.split(","))
        except (ValueError, TypeError):
            continue
        w = abs(x2 - x1)
        h = abs(y2 - y1)
        records.append({
            "image_id":  props.get("image_id", ""),
            "class_id":  int(props.get("type_id", -1)),
            "px_x1": min(x1, x2), "px_y1": min(y1, y2),
            "px_x2": max(x1, x2), "px_y2": max(y1, y2),
            "px_width": w, "px_height": h, "px_area": w * h,
        })

    df = pd.DataFrame(records)
    print(f"  {len(df):,} bounding boxes from "
          f"{df['image_id'].nunique()} images")
    return df

bbox_df = load_xview_geojson(GEOJSON_PATH)
bbox_df.head()


In [ ]:
# ==========================================================================
#  4.2   Class map & target selection
# ==========================================================================
XVIEW_CLASSES = {
    11:'Fixed-wing Aircraft',12:'Small Aircraft',13:'Cargo Plane',
    15:'Helicopter',17:'Passenger Vehicle',18:'Small Car',
    19:'Bus',20:'Pickup Truck',21:'Utility Truck',23:'Truck',
    24:'Cargo Truck',25:'Truck w/Box',26:'Truck Tractor',27:'Trailer',
    28:'Truck w/Flatbed',29:'Truck w/Liquid',32:'Crane Truck',
    33:'Railway Vehicle',34:'Passenger Car',35:'Cargo Car',
    36:'Flat Car',37:'Tank Car',38:'Locomotive',40:'Maritime Vessel',
    41:'Motorboat',42:'Sailboat',44:'Tugboat',45:'Barge',
    47:'Fishing Vessel',49:'Ferry',50:'Yacht',51:'Container Ship',
    52:'Oil Tanker',53:'Engineering Vehicle',54:'Tower Crane',
    55:'Container Crane',56:'Reach Stacker',57:'Straddle Carrier',
    59:'Mobile Crane',60:'Dump Truck',61:'Haul Truck',
    62:'Scraper/Tractor',63:'Front Loader/Bulldozer',64:'Excavator',
    65:'Cement Mixer',66:'Ground Grader',71:'Hut/Tent',72:'Shed',
    73:'Building',74:'Aircraft Hangar',76:'Damaged Building',
    77:'Facility',79:'Construction Site',83:'Vehicle Lot',
    84:'Helipad',86:'Storage Tank',89:'Shipping Container Lot',
    91:'Shipping Container',93:'Pylon',94:'Tower',
}

TARGET_CLASSES = {18:'Small Car', 17:'Passenger Vehicle',
                  20:'Pickup Truck', 21:'Utility Truck'}
CLASS_TO_IDX   = {18: 0, 17: 1, 20: 2, 21: 3}

bbox_df["class_name"] = bbox_df["class_id"].map(XVIEW_CLASSES).fillna("Unknown")
bbox_df["is_target"]  = bbox_df["class_id"].isin(TARGET_CLASSES)

# Filter to small objects matching the paper definition
small_df = bbox_df[
    bbox_df["is_target"]
    & (bbox_df["px_area"] >= MIN_OBJECT_PIXELS)
    & (bbox_df["px_area"] <= MAX_OBJECT_PIXELS)
].copy()

print("Target classes (Paper Table 3):\n")
for cid, cname in TARGET_CLASSES.items():
    s = small_df[small_df["class_id"] == cid]
    if len(s):
        print(f"  {cname:22s}  ID={cid:2d}  "
              f"{len(s):>8,} objects  {s['image_id'].nunique():>4} images  "
              f"area {s['px_area'].min():.0f}–{s['px_area'].max():.0f}")

print(f"\nTotal small objects : {len(small_df):,}")
print(f"Across {small_df['image_id'].nunique()} images")


---
## 5. Size & Aspect-Ratio Analysis (Paper Figure 3)

The paper shows object-size distribution on a log scale and aspect-ratio
distribution peaking at 0.7–0.8.


In [ ]:
# ==========================================================================
#  5.1   Reproduce Figure 3
# ==========================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) size distribution
ax = axes[0]
bins_s = np.logspace(0, np.log10(100_000), 20)
ax.hist(bbox_df["px_area"].clip(upper=100_000), bins=bins_s,
        alpha=.7, color="steelblue", edgecolor="w", label="All objects")
ax.hist(small_df["px_area"].clip(upper=100_000), bins=bins_s,
        alpha=.7, color="indianred", edgecolor="w", label="Target small")
ax.axvline(1000, color="red", ls="--", alpha=.6, label="1 000 px")
ax.set_xscale("log"); ax.set_xlabel("Area (px)"); ax.set_ylabel("Count")
ax.set_title("(a) Object Size Distribution"); ax.legend(fontsize=9)

# (b) aspect ratio
ax = axes[1]
short = np.minimum(bbox_df["px_width"], bbox_df["px_height"])
long  = np.maximum(bbox_df["px_width"], bbox_df["px_height"])
ar    = (short / long.replace(0, np.nan)).dropna()
ts = np.minimum(small_df["px_width"], small_df["px_height"])
tl = np.maximum(small_df["px_width"], small_df["px_height"])
ar_t = (ts / tl.replace(0, np.nan)).dropna()
b = np.arange(0, 1.05, .1)
ax.hist(ar, bins=b, alpha=.7, color="steelblue", edgecolor="w", label="All")
ax.hist(ar_t, bins=b, alpha=.7, color="indianred", edgecolor="w", label="Target")
ax.set_xlabel("Aspect Ratio"); ax.set_ylabel("Count")
ax.set_title("(b) Aspect Ratio Distribution"); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PATHS["PLOTS_DIR"] / "fig3_size_aspect.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved fig3_size_aspect.png")


In [ ]:
# ==========================================================================
#  5.2   Paper Table 3 reproduction
# ==========================================================================
rows = []
for cid, cn in TARGET_CLASSES.items():
    s = small_df[small_df["class_id"] == cid]
    if len(s):
        rows.append({"Class": cn, "Images": s["image_id"].nunique(),
                     "Objects": len(s), "Min": int(s["px_area"].min()),
                     "Median": int(s["px_area"].median()),
                     "Max": int(s["px_area"].max())})
t3 = pd.DataFrame(rows)
print(t3.to_string(index=False))
t3.to_csv(PATHS["METRICS_DIR"] / "table3.csv", index=False)


In [ ]:
# ==========================================================================
#  5.3   Class & density plots
# ==========================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# class counts
cc = small_df["class_id"].map(TARGET_CLASSES).value_counts()
cc.plot.bar(ax=axes[0], color=["#2196F3","#FF9800","#4CAF50","#F44336"],
            edgecolor="w")
axes[0].set_title("Class Distribution"); axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=30)
for i,v in enumerate(cc.values):
    axes[0].text(i, v+200, f"{v:,}", ha="center", fontsize=9)

opi = small_df.groupby("image_id").size()
axes[1].hist(opi, bins=50, color="steelblue", edgecolor="w")
axes[1].axvline(opi.median(), color="red", ls="--",
                label=f"Median={opi.median():.0f}")
axes[1].set_title("Objects per Image"); axes[1].set_xlabel("Count")
axes[1].legend()
plt.tight_layout()
plt.savefig(PATHS["PLOTS_DIR"] / "class_distribution.png",
            dpi=150, bbox_inches="tight")
plt.show()


---
## 6. Image Tiling (Paper Section 4.1)

> "Images are cropped into tiles of 512 x 512 ... rounding up ensures
> complete coverage with minimal overlap."

Boxes clipped to tile boundaries; tiles with no target objects are skipped.
Results are **cached** — re-running this cell is fast.


In [ ]:
# ==========================================================================
#  6.1   Tiling helpers
# ==========================================================================
def tile_positions(h, w, ts=512):
    nr, nc = int(np.ceil(h/ts)), int(np.ceil(w/ts))
    out = []
    for r in range(nr):
        for c in range(nc):
            y = r*(h-ts)//(nr-1) if nr>1 else 0
            x = c*(w-ts)//(nc-1) if nc>1 else 0
            out.append((min(y, max(0,h-ts)), min(x, max(0,w-ts))))
    return out

def clip_box(box, ty, tx, ts, min_ratio=.2):
    x1 = max(box["px_x1"]-tx, 0); y1 = max(box["px_y1"]-ty, 0)
    x2 = min(box["px_x2"]-tx, ts); y2 = min(box["px_y2"]-ty, ts)
    cw, ch = x2-x1, y2-y1
    if cw<=0 or ch<=0:
        return None
    if box["px_area"]>0 and cw*ch/box["px_area"] < min_ratio:
        return None
    return {"x1":x1,"y1":y1,"x2":x2,"y2":y2,
            "w":cw,"h":ch,"a":cw*ch,"cid":box["class_id"]}

print("Tiling helpers ready.")


In [ ]:
# ==========================================================================
#  6.2   Run tiling pipeline (cached)
# ==========================================================================
def run_tiling(images_dir, annotations, paths, tile_size=512):
    img_out = paths["TILES_DIR"] / "images"
    lbl_out = paths["TILES_DIR"] / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    manifest_csv = paths["TILES_DIR"] / "manifest.csv"

    # Cache check
    if manifest_csv.exists():
        mf = pd.read_csv(manifest_csv)
        existing = len(list(img_out.glob("*.png")))
        if existing >= len(mf) and existing > 50:
            print(f"Tile cache valid — {existing} tiles already exist.")
            return mf

    target_imgs = annotations["image_id"].unique()
    print(f"Tiling {len(target_imgs)} images ...")
    records = []

    for img_id in tqdm(target_imgs, desc="Tiling"):
        # locate file
        ip = images_dir / str(img_id)
        if not ip.exists():
            stem = str(img_id).split(".")[0]
            for ext in [".tif",".tiff",".png"]:
                c = images_dir / (stem+ext)
                if c.exists(): ip=c; break
        if not ip.exists():
            continue
        img = cv2.imread(str(ip))
        if img is None:
            continue
        h, w = img.shape[:2]
        iboxes = annotations[annotations["image_id"]==img_id]
        if len(iboxes)==0:
            continue
        for idx,(ty,tx) in enumerate(tile_positions(h,w,tile_size)):
            tile = img[ty:ty+tile_size, tx:tx+tile_size]
            if tile.shape[0]<tile_size or tile.shape[1]<tile_size:
                pad = np.zeros((tile_size,tile_size,3), dtype=np.uint8)
                pad[:tile.shape[0],:tile.shape[1]] = tile
                tile = pad
            tboxes = []
            for _,bx in iboxes.iterrows():
                cb = clip_box(bx, ty, tx, tile_size)
                if cb: tboxes.append(cb)
            if not tboxes:
                continue
            tn = f"{Path(img_id).stem}_t{idx:04d}"
            cv2.imwrite(str(img_out/f"{tn}.png"), tile)
            lines = []
            for b in tboxes:
                cx=(b["x1"]+b["x2"])/2/tile_size
                cy=(b["y1"]+b["y2"])/2/tile_size
                bw=b["w"]/tile_size; bh=b["h"]/tile_size
                ci=CLASS_TO_IDX.get(b["cid"],0)
                lines.append(f"{ci} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            (lbl_out/f"{tn}.txt").write_text("\n".join(lines))
            records.append({"tile":tn,"src":img_id,"n_obj":len(tboxes)})

    mf = pd.DataFrame(records)
    mf.to_csv(manifest_csv, index=False)
    print(f"Done: {len(mf)} tiles, {mf['n_obj'].sum():,} objects")
    return mf

manifest = run_tiling(IMAGES_DIR, small_df, PATHS, TILE_SIZE)
print(f"Manifest shape: {manifest.shape}")


---
## 7. Train / Validation Split

Split at the **source-image level** (70/30) to prevent leakage.


In [ ]:
# ==========================================================================
#  7.1   Split & organise YOLO layout
# ==========================================================================
def split_and_organise(manifest, paths, ratio=0.7, seed=42):
    yolo = paths["YOLO_DATASET"]
    yaml_p = yolo / "dataset.yaml"

    # Cache check
    tr_dir = yolo/"images"/"train"
    vl_dir = yolo/"images"/"val"
    tr_dir.mkdir(parents=True, exist_ok=True)
    vl_dir.mkdir(parents=True, exist_ok=True)
    (yolo/"labels"/"train").mkdir(parents=True, exist_ok=True)
    (yolo/"labels"/"val").mkdir(parents=True, exist_ok=True)

    if yaml_p.exists() and len(list(tr_dir.glob("*.png"))) > 50:
        n_tr = len(list(tr_dir.glob("*.png")))
        n_vl = len(list(vl_dir.glob("*.png")))
        print(f"YOLO dataset cached: {n_tr} train, {n_vl} val")
        return yaml_p

    rng = np.random.RandomState(seed)
    srcs = manifest["src"].unique()
    rng.shuffle(srcs)
    n = int(len(srcs)*ratio)
    tr_set = set(srcs[:n])

    def lnk(src, dst):
        if dst.exists(): return
        try: os.symlink(src.resolve(), dst)
        except OSError: shutil.copy2(src, dst)

    si = paths["TILES_DIR"]/"images"
    sl = paths["TILES_DIR"]/"labels"
    n_tr = n_vl = 0
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Organising"):
        t = row["tile"]
        split = "train" if row["src"] in tr_set else "val"
        ip = si/f"{t}.png"; lp = sl/f"{t}.txt"
        if ip.exists():
            lnk(ip, yolo/"images"/split/f"{t}.png")
        if lp.exists():
            lnk(lp, yolo/"labels"/split/f"{t}.txt")
        if split=="train": n_tr+=1
        else: n_vl+=1

    yd = {"path": str(yolo.resolve()), "train":"images/train",
          "val":"images/val", "nc":4,
          "names":{0:"Small_Car",1:"Passenger_Vehicle",
                   2:"Pickup_Truck",3:"Utility_Truck"}}
    with open(yaml_p,"w") as f:
        yaml.dump(yd, f, default_flow_style=False)

    # Save split metadata
    meta = {"seed":seed,"ratio":ratio,"train_tiles":n_tr,"val_tiles":n_vl,
            "train_images":int(n),"val_images":int(len(srcs)-n)}
    with open(paths["METRICS_DIR"]/"split_info.json","w") as f:
        json.dump(meta, f, indent=2)

    print(f"YOLO dataset ready: {n_tr} train, {n_vl} val")
    return yaml_p

DATASET_YAML = split_and_organise(manifest, PATHS, TRAIN_RATIO, SEED)
print(f"dataset.yaml : {DATASET_YAML}")


---
## 8. YOLOv11 Training

### Paper Table 5 config
| LR | BS | Momentum | Decay | Augmentation |
|---|---|---|---|---|
| 1.01e-4 | 8 | 0.89 | 5e-4 | Horizontal flip p=0.5 |

### Checkpointing
- `best.pt` saved when mAP improves
- `last.pt` saved every epoch
- Intermediate every 5 epochs
- Set `RESUME_TRAINING = True` to continue from last checkpoint


In [ ]:
# ==========================================================================
#  8.1   Training arguments (paper-aligned)
# ==========================================================================
train_args = dict(
    data         = str(DATASET_YAML),
    epochs       = EPOCHS,
    batch        = BATCH_SIZE,
    imgsz        = IMG_SIZE,
    lr0          = LEARNING_RATE,
    lrf          = 0.01,
    momentum     = MOMENTUM,
    weight_decay = WEIGHT_DECAY,
    # --- Augmentation: paper uses ONLY horizontal flip ---
    flipud=0., fliplr=FLIP_PROB, mosaic=0., mixup=0.,
    scale=0., hsv_h=0., hsv_s=0., hsv_v=0.,
    # --- Optimiser ---
    optimizer     = "SGD",
    seed          = SEED,
    deterministic = True,
    workers       = 4,
    patience      = 20,
    # --- Saving ---
    project     = str(PATHS["CHECKPOINTS_DIR"]),
    name        = "yolov11_xview",
    exist_ok    = True,
    save        = True,
    save_period = 5,
    plots       = True,
    verbose     = True,
)

print("Training config:")
for k in ["epochs","batch","imgsz","lr0","momentum","optimizer"]:
    print(f"  {k:16s}: {train_args[k]}")


In [ ]:
# ==========================================================================
#  8.2   Resume logic
# ==========================================================================
EXP_DIR     = PATHS["CHECKPOINTS_DIR"] / "yolov11_xview"
WEIGHTS_DIR = EXP_DIR / "weights"

resume_from = None
if RESUME_TRAINING and WEIGHTS_DIR.exists():
    last = WEIGHTS_DIR / "last.pt"
    if last.exists():
        resume_from = str(last)
        print(f"Will RESUME from : {last}")
    else:
        pts = sorted(WEIGHTS_DIR.glob("epoch*.pt"), reverse=True)
        if pts:
            resume_from = str(pts[0])
            print(f"Will RESUME from : {pts[0]}")

if resume_from is None:
    print("No checkpoint found — training will start from scratch.")


In [ ]:
# ==========================================================================
#  8.3   TRAIN
# ==========================================================================
# Runtime:  T4 ~2-4 h  |  RTX 4090 ~1-2 h  |  CPU: not recommended
# Safe to interrupt — set RESUME_TRAINING=True and re-run.

def train(variant, args, resume_path):
    if resume_path:
        print(f"Resuming from {resume_path}")
        model = YOLO(resume_path)
        results = model.train(resume=True)
    else:
        print(f"Training {variant} from scratch")
        model = YOLO(variant)
        results = model.train(**args)
    return model, results

if not SKIP_TRAINING:
    model, results = train(YOLO_VARIANT, train_args, resume_from)
    print("Training complete.")
else:
    model = None
    print("SKIP_TRAINING=True — skipping to evaluation.")


---
## 9. Evaluation (Paper Table 13 metrics)

AP50, AP50:95, Precision, Recall, F1.


In [ ]:
# ==========================================================================
#  9.1   Evaluate
# ==========================================================================
def evaluate(paths, dataset_yaml, conf=0.25):
    best = paths["CHECKPOINTS_DIR"] / "yolov11_xview" / "weights" / "best.pt"
    if not best.exists():
        print(f"No model at {best}.  Train first.")
        return None
    print(f"Evaluating {best}")
    m = YOLO(str(best))
    r = m.val(data=str(dataset_yaml), imgsz=IMG_SIZE,
              conf=conf, split="val", plots=True, verbose=True)
    met = {
        "AP50":      float(r.box.map50)*100,
        "AP50_95":   float(r.box.map)*100,
        "Precision": float(r.box.mp)*100,
        "Recall":    float(r.box.mr)*100,
    }
    p, rc = met["Precision"]/100, met["Recall"]/100
    met["F1"] = 2*p*rc/(p+rc)*100 if (p+rc)>0 else 0

    print("\n" + "="*60)
    print("  RESULTS")
    print("="*60)
    for k,v in met.items():
        print(f"  {k:12s}: {v:6.2f}%")

    paper = {"AP50":69.47, "AP50_95":29.58, "F1":73.97}
    print("\n  vs Paper (YOLOv11 on xView):")
    for k,pv in paper.items():
        ov=met.get(k,0); d=ov-pv
        print(f"  {k:12s}: Ours={ov:6.2f}%  Paper={pv:6.2f}%  {'+' if d>=0 else ''}{d:.2f}%")

    pd.DataFrame([met]).to_csv(paths["METRICS_DIR"]/"results.csv", index=False)
    return met

eval_met = evaluate(PATHS, DATASET_YAML, CONF_THRESHOLD)


---
## 10. Visual Diagnostics


In [ ]:
# ==========================================================================
#  10.1  Prediction overlay
# ==========================================================================
def viz_preds(paths, n=6):
    best = paths["CHECKPOINTS_DIR"]/"yolov11_xview"/"weights"/"best.pt"
    if not best.exists():
        print("No model."); return
    m = YOLO(str(best))
    vd = paths["YOLO_DATASET"]/"images"/"val"
    imgs = list(vd.glob("*.png"))
    if not imgs: print("No val images."); return
    sel = random.sample(imgs, min(n, len(imgs)))
    rows = (len(sel)+2)//3
    fig, axes = plt.subplots(rows, 3, figsize=(20, 7*rows))
    axes = np.array(axes).flatten()
    for i, ip in enumerate(sel):
        if i>=len(axes): break
        ax=axes[i]
        img=cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
        lp = paths["YOLO_DATASET"]/"labels"/"val"/(ip.stem+".txt")
        ng=0
        if lp.exists():
            for ln in lp.read_text().strip().split("\n"):
                p=ln.split()
                if len(p)==5:
                    _,cx,cy,w,h=map(float,p)
                    cv2.rectangle(img,
                        (int((cx-w/2)*512),int((cy-h/2)*512)),
                        (int((cx+w/2)*512),int((cy+h/2)*512)),(0,255,0),2)
                    ng+=1
        r=m.predict(str(ip), conf=.25, verbose=False)
        np_=0
        if r and len(r[0].boxes):
            for b in r[0].boxes:
                x1,y1,x2,y2=b.xyxy[0].cpu().numpy().astype(int)
                cv2.rectangle(img,(x1,y1),(x2,y2),(255,0,0),2); np_+=1
        ax.imshow(img); ax.set_title(f"GT:{ng}  Pred:{np_}"); ax.axis("off")
    for j in range(i+1, len(axes)): axes[j].axis("off")
    plt.suptitle("Green=GT  Red=Pred", fontsize=14)
    plt.tight_layout()
    plt.savefig(paths["PLOTS_DIR"]/"predictions.png", dpi=150, bbox_inches="tight")
    plt.show()

viz_preds(PATHS)


In [ ]:
# ==========================================================================
#  10.2  Training curves
# ==========================================================================
def plot_curves(paths):
    csv = paths["CHECKPOINTS_DIR"]/"yolov11_xview"/"results.csv"
    if not csv.exists(): print("No results.csv"); return
    df = pd.read_csv(csv); df.columns=[c.strip() for c in df.columns]
    fig, axes = plt.subplots(2, 3, figsize=(18,10))
    specs = [("train/box_loss","val/box_loss","Box Loss"),
             ("train/cls_loss","val/cls_loss","Cls Loss"),
             ("train/dfl_loss","val/dfl_loss","DFL Loss")]
    for i,(t,v,ttl) in enumerate(specs):
        ax=axes[0,i]
        if t in df: ax.plot(df["epoch"],df[t],label="Train",color="steelblue")
        if v in df: ax.plot(df["epoch"],df[v],label="Val",color="indianred")
        ax.set_title(ttl); ax.legend(); ax.grid(alpha=.3)
    ax=axes[1,0]
    for c in [x for x in df.columns if "map" in x.lower()]:
        ax.plot(df["epoch"],df[c],label=c)
    ax.set_title("mAP"); ax.legend(fontsize=8); ax.grid(alpha=.3)
    ax=axes[1,1]
    if "metrics/precision(B)" in df:
        ax.plot(df["epoch"],df["metrics/precision(B)"],label="P")
        ax.plot(df["epoch"],df["metrics/recall(B)"],label="R")
    ax.set_title("Precision & Recall"); ax.legend(); ax.grid(alpha=.3)
    ax=axes[1,2]
    for c in [x for x in df.columns if "lr" in x.lower()]:
        ax.plot(df["epoch"],df[c],label=c)
    ax.set_title("LR"); ax.legend(fontsize=8); ax.grid(alpha=.3)
    for a in axes.flatten(): a.set_xlabel("Epoch")
    plt.suptitle("Training Curves", fontsize=14)
    plt.tight_layout()
    plt.savefig(paths["PLOTS_DIR"]/"training_curves.png",dpi=150,bbox_inches="tight")
    plt.show()

plot_curves(PATHS)


---
## 11. Error Analysis

Size-dependent and density-dependent recall, plus overfitting check.


In [ ]:
# ==========================================================================
#  11.1  Error analysis by size & density
# ==========================================================================
def error_analysis(paths, n=200):
    best = paths["CHECKPOINTS_DIR"]/"yolov11_xview"/"weights"/"best.pt"
    if not best.exists(): print("No model."); return
    m = YOLO(str(best))
    vimg = paths["YOLO_DATASET"]/"images"/"val"
    vlbl = paths["YOLO_DATASET"]/"labels"/"val"
    imgs = list(vimg.glob("*.png"))
    if not imgs: print("No val images."); return
    samp = random.sample(imgs, min(n,len(imgs)))

    S = {"tp":0,"fp":0,"fn":0,"gt":0,"pred":0,
         "sz": defaultdict(lambda:{"gt":0,"tp":0}),
         "dn": defaultdict(lambda:{"gt":0,"tp":0})}

    for ip in tqdm(samp, desc="Analysing"):
        lp = vlbl/(ip.stem+".txt")
        gt=[]
        if lp.exists():
            for ln in lp.read_text().strip().split("\n"):
                p=ln.split()
                if len(p)==5:
                    _,cx,cy,w,h=map(float,p)
                    gt.append({"cx":cx,"cy":cy,"w":w,"h":h,"a":w*h*512*512})
        r=m.predict(str(ip),conf=.25,verbose=False)
        pds=[]
        if r and len(r[0].boxes):
            for b in r[0].boxes:
                x1,y1,x2,y2=b.xyxy[0].cpu().numpy()/512
                pds.append({"x1":x1,"y1":y1,"x2":x2,"y2":y2})
        S["gt"]+=len(gt); S["pred"]+=len(pds)
        mg=set(); mp=set()
        for pi,p in enumerate(pds):
            bi,bv=(-1,0)
            for gi,g in enumerate(gt):
                if gi in mg: continue
                gx1=g["cx"]-g["w"]/2; gy1=g["cy"]-g["h"]/2
                gx2=g["cx"]+g["w"]/2; gy2=g["cy"]+g["h"]/2
                ix=max(0,min(p["x2"],gx2)-max(p["x1"],gx1))
                iy=max(0,min(p["y2"],gy2)-max(p["y1"],gy1))
                inter=ix*iy
                un=(p["x2"]-p["x1"])*(p["y2"]-p["y1"])+g["w"]*g["h"]-inter
                iou=inter/max(un,1e-9)
                if iou>bv: bv=iou; bi=gi
            if bv>=.5: mg.add(bi); mp.add(pi); S["tp"]+=1
        S["fp"]+=len(pds)-len(mp); S["fn"]+=len(gt)-len(mg)
        for gi,g in enumerate(gt):
            a=g["a"]
            b="<100" if a<100 else "100-300" if a<300 else "300-600" if a<600 else "600+"
            S["sz"][b]["gt"]+=1
            if gi in mg: S["sz"][b]["tp"]+=1
        d="sparse" if len(gt)<=5 else "moderate" if len(gt)<=20 else "dense"
        S["dn"][d]["gt"]+=len(gt); S["dn"][d]["tp"]+=len(mg)

    pr=S["tp"]/max(S["tp"]+S["fp"],1)
    rc=S["tp"]/max(S["tp"]+S["fn"],1)
    f1=2*pr*rc/max(pr+rc,1e-9)
    print(f"\nPrecision={pr:.3f}  Recall={rc:.3f}  F1={f1:.3f}")
    print(f"TP={S['tp']}  FP={S['fp']}  FN={S['fn']}")
    print("\nBy size:")
    for b in ["<100","100-300","300-600","600+"]:
        s=S["sz"][b]
        if s["gt"]: print(f"  {b:10s} {s['gt']:5d} GT  recall={s['tp']/s['gt']:.3f}")
    print("By density:")
    for d in ["sparse","moderate","dense"]:
        s=S["dn"][d]
        if s["gt"]: print(f"  {d:10s} {s['gt']:5d} GT  recall={s['tp']/s['gt']:.3f}")
    with open(paths["METRICS_DIR"]/"error_analysis.json","w") as f:
        json.dump({"pr":pr,"rc":rc,"f1":f1},f,indent=2)

error_analysis(PATHS)


In [ ]:
# ==========================================================================
#  11.2  Overfitting check
# ==========================================================================
def check_overfit(paths):
    csv = paths["CHECKPOINTS_DIR"]/"yolov11_xview"/"results.csv"
    if not csv.exists(): print("No log."); return
    df = pd.read_csv(csv); df.columns=[c.strip() for c in df.columns]
    if "train/box_loss" not in df or "val/box_loss" not in df: return
    fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
    a1.plot(df["epoch"],df["train/box_loss"],label="Train",lw=2,color="steelblue")
    a1.plot(df["epoch"],df["val/box_loss"],label="Val",lw=2,color="indianred")
    a1.set_title("Train vs Val Loss"); a1.legend(); a1.grid(alpha=.3)
    if len(df)>20:
        l20=df.tail(20)
        ts=np.polyfit(range(20),l20["train/box_loss"].values,1)[0]
        vs=np.polyfit(range(20),l20["val/box_loss"].values,1)[0]
        msg="OVERFITTING" if ts<0 and vs>0 else "OK"
        col="red" if msg=="OVERFITTING" else "green"
        a1.text(.5,.95,msg,transform=a1.transAxes,ha="center",
                va="top",fontsize=12,color=col,fontweight="bold")
    gap=df["val/box_loss"]-df["train/box_loss"]
    a2.plot(df["epoch"],gap,color="purple",lw=2)
    a2.axhline(0,color="gray",ls="--",alpha=.5)
    a2.fill_between(df["epoch"],0,gap,alpha=.2,color="purple")
    a2.set_title("Generalisation Gap"); a2.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(paths["PLOTS_DIR"]/"overfitting.png",dpi=150,bbox_inches="tight")
    plt.show()

check_overfit(PATHS)


---
## 12. Comparative Framework

Scaffolds for RT-DETR and Faster R-CNN.  Uncomment to train.


In [ ]:
# ==========================================================================
#  12.1  RT-DETR scaffold (Paper: AP50=68.20)
# ==========================================================================
def train_rtdetr():
    m = YOLO("rtdetr-l.pt")
    m.train(data=str(DATASET_YAML), epochs=EPOCHS, batch=3,
            imgsz=IMG_SIZE, lr0=7.68e-4, momentum=.92,
            weight_decay=9.88e-4, fliplr=FLIP_PROB, mosaic=0.,
            project=str(PATHS["CHECKPOINTS_DIR"]),
            name="rtdetr_xview", exist_ok=True, seed=SEED)
    return m

# Uncomment:  rtdetr = train_rtdetr()
print("RT-DETR scaffold ready.")


In [ ]:
# ==========================================================================
#  12.2  Faster R-CNN scaffold (Paper: AP50=61.65)
# ==========================================================================
def create_faster_rcnn(nc=5, anc_max=90):
    from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.rpn import AnchorGenerator
    m = fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
    sizes = tuple((s,) for s in [16,32,64,anc_max])
    ar = ((0.75,0.85,0.95),)*len(sizes)
    m.rpn.anchor_generator = AnchorGenerator(sizes=sizes, aspect_ratios=ar)
    inf = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(inf, nc)
    return m

# Uncomment:  frcnn = create_faster_rcnn()
print("Faster R-CNN scaffold ready.")


In [ ]:
# ==========================================================================
#  12.3  Comparison table
# ==========================================================================
def comparison_table(paths):
    res = {}
    yaml_p = str(paths["YOLO_DATASET"]/"dataset.yaml")
    for nm, sub in [("YOLOv11","yolov11_xview"),("RT-DETR","rtdetr_xview")]:
        bp = paths["CHECKPOINTS_DIR"]/sub/"weights"/"best.pt"
        if bp.exists():
            m=YOLO(str(bp))
            r=m.val(data=yaml_p,imgsz=IMG_SIZE,conf=CONF_THRESHOLD,verbose=False)
            res[nm]= {"AP50":float(r.box.map50)*100,"AP50_95":float(r.box.map)*100}
    paper = {
        "YOLOv11 (paper)":     {"AP50":69.47,"AP50_95":29.58},
        "RT-DETR (paper)":     {"AP50":68.20,"AP50_95":28.15},
        "Faster R-CNN (paper)":{"AP50":61.65,"AP50_95":23.32},
        "Cascade R-CNN (paper)":{"AP50":55.60,"AP50_95":21.53},
        "SSD (paper)":         {"AP50":13.15,"AP50_95": 3.35},
    }
    res.update(paper)
    df = pd.DataFrame(res).T
    print(df.to_string())
    df.to_csv(paths["METRICS_DIR"]/"comparison.csv")
    return df

comparison_table(PATHS)


---
## 13. Scale-Dependent Performance (Paper Section 4.5)


In [ ]:
# ==========================================================================
#  13.1  Recall by size bin
# ==========================================================================
def scale_plot(paths, n=200):
    best=paths["CHECKPOINTS_DIR"]/"yolov11_xview"/"weights"/"best.pt"
    if not best.exists(): print("No model."); return
    m=YOLO(str(best))
    vimg=paths["YOLO_DATASET"]/"images"/"val"
    vlbl=paths["YOLO_DATASET"]/"labels"/"val"
    imgs=list(vimg.glob("*.png"))
    samp=random.sample(imgs, min(n,len(imgs)))
    sd=defaultdict(lambda:{"tp":0,"tot":0})
    for ip in tqdm(samp, desc="Scale"):
        lp=vlbl/(ip.stem+".txt")
        gt=[]
        if lp.exists():
            for ln in lp.read_text().strip().split("\n"):
                p=ln.split()
                if len(p)==5:
                    _,cx,cy,w,h=map(float,p)
                    gt.append({"cx":cx,"cy":cy,"w":w,"h":h,"a":w*h*512*512})
        r=m.predict(str(ip),conf=.25,verbose=False)
        pds=[]
        if r and len(r[0].boxes):
            for b in r[0].boxes:
                x1,y1,x2,y2=b.xyxy[0].cpu().numpy()/512
                pds.append({"x1":x1,"y1":y1,"x2":x2,"y2":y2})
        mg=set()
        for g in gt:
            bv,bp=0,-1
            for pi,p in enumerate(pds):
                if pi in mg: continue
                gx1=g["cx"]-g["w"]/2; gy1=g["cy"]-g["h"]/2
                gx2=g["cx"]+g["w"]/2; gy2=g["cy"]+g["h"]/2
                ix=max(0,min(p["x2"],gx2)-max(p["x1"],gx1))
                iy=max(0,min(p["y2"],gy2)-max(p["y1"],gy1))
                inter=ix*iy
                un=(p["x2"]-p["x1"])*(p["y2"]-p["y1"])+g["w"]*g["h"]-inter
                iou=inter/max(un,1e-9)
                if iou>bv: bv=iou;bp=pi
            a=g["a"]
            bn="<50" if a<50 else "50-150" if a<150 else "150-300" if a<300 else "300-500" if a<500 else "500+"
            sd[bn]["tot"]+=1
            if bv>=.5 and bp>=0: sd[bn]["tp"]+=1; mg.add(bp)
    order=["<50","50-150","150-300","300-500","500+"]
    recs=[sd[b]["tp"]/max(sd[b]["tot"],1) for b in order]
    tots=[sd[b]["tot"] for b in order]
    fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
    a1.bar(order,recs,color=plt.cm.RdYlGn(np.array(recs)),edgecolor="w")
    a1.set_ylim(0,1.05); a1.set_ylabel("Recall@0.50"); a1.set_xlabel("Area (px)")
    a1.set_title("Recall by Object Size")
    for i,(r,t) in enumerate(zip(recs,tots)):
        a1.text(i,r+.02,f"{r:.2f}\n(n={t})",ha="center",fontsize=9)
    a2.bar(order,tots,color="steelblue",edgecolor="w")
    a2.set_ylabel("GT Count"); a2.set_xlabel("Area (px)")
    a2.set_title("Distribution")
    plt.tight_layout()
    plt.savefig(paths["PLOTS_DIR"]/"scale_perf.png",dpi=150,bbox_inches="tight")
    plt.show()

scale_plot(PATHS)


---
## 14. Conclusion

### Reproduced from the paper
| Aspect | Status |
|---|---|
| 512x512 tiling, GeoJSON parsing | Done |
| Small-object class filter (Table 3) | Done |
| Size & aspect-ratio analysis (Figure 3) | Done |
| YOLOv11 training, paper hyper-params | Done |
| AP50, AP50:95, F1 (Table 13) | Done |
| Error analysis (size, density) | Done |
| Overfitting check | Done |
| RT-DETR, Faster R-CNN scaffolds | Scaffolded |

### Differences
- Single 70/30 split (paper: 6-fold CV)
- YOLOv11 nano/small (paper: ~25 M params — use `yolo11m.pt`)
- No SkySat/DOTA transfer-learning

### Next steps
1. `YOLO_VARIANT = "yolo11m.pt"` for paper-scale params
2. 6-fold CV for statistical rigour
3. Train RT-DETR / Faster R-CNN from scaffolds
4. Anchor-size sensitivity (Paper Table 7)


In [ ]:
# ==========================================================================
#  14.1  Save experiment summary
# ==========================================================================
summary = {
    "paper":       "Yuan et al. 2026, ESWA 307, 131061",
    "dataset":     DATASET_HANDLE,
    "model":       YOLO_VARIANT,
    "run_env":     RUN_ENV,
    "tile_size":   TILE_SIZE,
    "max_obj_px":  MAX_OBJECT_PIXELS,
    "classes":     list(TARGET_CLASSES.values()),
    "train_ratio": TRAIN_RATIO,
    "epochs":      EPOCHS,
    "batch_size":  BATCH_SIZE,
    "lr":          LEARNING_RATE,
    "timestamp":   datetime.now().isoformat(),
    "gpu":         HW["name"],
}
with open(PATHS["METRICS_DIR"]/"experiment_summary.json","w") as f:
    json.dump(summary, f, indent=2)

print("Experiment summary saved.")
print(f"\nAll outputs at: {PATHS['OUTPUT_ROOT']}")
for k in ["CHECKPOINTS_DIR","PLOTS_DIR","METRICS_DIR","PREDICTIONS_DIR",
          "TILES_DIR","YOLO_DATASET"]:
    p = PATHS[k]
    print(f"  {k:18s}: {p}")
